# Inference recording: tracking error

Same error plots as `visualize_trajectory.ipynb`, but computed from a **recorded rollout**
instead of replaying a dataset episode on the robot. The recording in
`blocksuite/inference_recordings/` was written by the `rcs` `StorageWrapper`
(see `python/rcs/envs/storage_wrapper.py`) while `vlagent_rcs_client.py` drove the real
FR3 + Robotiq 2F-85 from the policy server, so every row already contains **both** sides of
the comparison the replay notebook had to reconstruct:

| column | meaning |
| --- | --- |
| `env_action[robot]["joints"]` | absolute joint target (rad) commanded at this step — the analogue of the dataset's `action.joint_target` |
| `obs[robot]["joints"]` | joint position measured *after* that step was executed — the analogue of the replay's `q_robot` |
| `env_action/obs[robot]["gripper"]` | binary gripper command / reading (0 = closed, 1 = open) |
| `obs[robot]["robot_state"]` | full `franka::RobotState` (`q`, `dq`, `tau_J`, `O_T_EE`, ...) |
| `info[robot]["limited_absolute_action"]` | the command after the `LimitedAbsoluteAction` wrapper clipped it |
| `obs["frames"][cam]["rgb"]["data"]` | JPEG-encoded camera frame |

Because `StorageWrapper.step()` stores the observation returned by `env.step(action)`, row `k`
pairs the command sent at step `k` with the state that resulted from it. The tracking error is
therefore `obs.joints - env_action.joints`, the exact quantity the replay notebook printed as
`|robot - action|`.

Note: `action` is `None` in row 0 and lags one step behind `env_action` (the wrapper stores the
*previous* action there), so `env_action` is used throughout.

In [ ]:
import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

RECORDING_DIR = Path("/home/yejin/code/robot-control-stack/blocksuite/inference_recordings2")
ROBOT_NAME = "right"

Q_HOME = np.array([0.0, -np.pi / 4, 0.0, -3 * np.pi / 4, 0.0, np.pi / 2, 0.0])

# The recording is sharded: StorageWrapper flushes a new `part-{i}-{writer_id}.parquet`
# every `batch_size` steps, so a single episode is spread over many files. Concatenate
# everything and group by `uuid` (one uuid == one episode).
files = sorted(RECORDING_DIR.glob("*.parquet"))
rec = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
rec = rec.sort_values(["uuid", "step"]).reset_index(drop=True)

episodes = rec.groupby("uuid").agg(
    frames=("step", "size"),
    first_step=("step", "min"),
    last_step=("step", "max"),
    success=("success", "max"),
    instruction=("instruction", "first"),
    t_start=("timestamp", "min"),
)
episodes["duration_s"] = rec.groupby("uuid")["timestamp"].max() - episodes["t_start"]

print(f"{len(files)} parquet parts, {len(rec)} rows, {rec['uuid'].nunique()} episode(s)")
episodes.drop(columns="t_start")

## Pick an episode

Change `EPISODE_UUID` (index into the table above) and re-run the cells below.

In [ ]:
EPISODE_UUID = rec["uuid"].unique()[0]

ep = rec[rec["uuid"] == EPISODE_UUID].sort_values("step").reset_index(drop=True)
assert (np.diff(ep["step"].to_numpy()) == 1).all(), "episode has missing steps"


def robot_field(df: pd.DataFrame, col: str, *keys: str) -> np.ndarray:
    """Stack a per-robot field of a struct column into a (T, D) array."""
    out = []
    for row in df[col]:
        value = row[ROBOT_NAME]
        for key in keys:
            value = value[key]
        out.append(np.asarray(value, dtype=np.float64))
    return np.stack(out)


t = ep["timestamp"].to_numpy()
t = t - t[0]                                              # seconds since episode start
dt = np.diff(t)
FPS = 1.0 / np.median(dt)

q_obs = robot_field(ep, "obs", "joints")                  # (T, 7) measured after the step
q_cmd = robot_field(ep, "env_action", "joints")           # (T, 7) absolute joint target sent
q_lim = robot_field(ep, "info", "limited_absolute_action")  # (T, 7) after action limiter
gripper_obs = robot_field(ep, "obs", "gripper").squeeze(-1)
gripper_cmd = robot_field(ep, "env_action", "gripper").squeeze(-1)

print(f"episode {EPISODE_UUID[:8]}: {len(ep)} steps, {t[-1]:.1f} s, "
      f"{FPS:.1f} fps (dt median {np.median(dt) * 1e3:.1f} ms, max {dt.max() * 1e3:.1f} ms)")
print(f"instruction: {ep['instruction'].iloc[0]!r}, success={bool(ep['success'].max())}")
print(f"max |env_action - limited_absolute_action| = {np.abs(q_cmd - q_lim).max():.6f} rad "
      "(0 -> the recorded command is already the limiter output)")

## Joint observations vs. commanded actions

Same layout as the `Dataset vs. robot joint comparison` figure of `visualize_trajectory.ipynb`:
per joint the commanded absolute target and the joint reading that came back from the robot.
There is no third trace here — the recording has no separate "dataset" reference, the policy
generated the commands online. Dashed grey = `q_home`.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 12), sharex=True)
axes = axes.ravel()

for j in range(7):
    ax = axes[j]
    ax.plot(t, q_cmd[:, j], label="commanded (env_action)", color="tab:orange", lw=1.0, ls="--")
    ax.plot(t, q_obs[:, j], label="robot reading (obs)", color="tab:green", lw=1.4)
    ax.axhline(Q_HOME[j], color="grey", ls=":", lw=1.5, label="q_home")
    ax.set_title(f"joint_{j}")
    ax.set_ylabel("rad")
    ax.grid(alpha=0.3)
    if j == 0:
        ax.legend(loc="best", fontsize=8)

ax = axes[7]
ax.plot(t, gripper_cmd, label="commanded", color="tab:orange", lw=1.0, ls="--")
ax.plot(t, gripper_obs, label="robot reading", color="tab:green", lw=1.4)
ax.set_title("gripper (1 = open, 0 = closed)")
ax.grid(alpha=0.3)
ax.legend(loc="best", fontsize=8)

for ax in axes[6:]:
    ax.set_xlabel("time [s]")
fig.suptitle(f"Episode {EPISODE_UUID[:8]}: commanded vs. measured joints", y=1.0)
fig.tight_layout()
plt.show()

## Tracking error

`error = obs.joints - env_action.joints`, i.e. the same `|robot - action|` statistic the replay
notebook printed, now per joint and over time. The dotted red lines mark
`±max_relative_movement` (5°): the `LimitedAbsoluteAction` wrapper clips every command to that
band around the *measured* joint state, so an error that sits flat on the line means the joint is
rate-saturated — the policy is asking for a faster motion than the wrapper lets through.

In [ ]:
err = q_obs - q_cmd                       # (T, 7) rad
abs_err = np.abs(err)
MAX_REL_MOVEMENT = np.deg2rad(5)          # LimitedAbsoluteAction cap used for this rollout

print(f"tracking error |robot - action|: mean={abs_err.mean():.4f} rad, max={abs_err.max():.4f} rad")
print(f"gripper mismatch: {np.mean(gripper_obs != gripper_cmd) * 100:.1f}% of steps")

err_table = pd.DataFrame(
    {
        "mean |err| [rad]": abs_err.mean(0),
        "rms [rad]": np.sqrt((err ** 2).mean(0)),
        "max |err| [rad]": abs_err.max(0),
        "max |err| [deg]": np.rad2deg(abs_err.max(0)),
        "% steps at limiter cap": 100 * np.isclose(abs_err, MAX_REL_MOVEMENT, atol=1e-9).mean(0),
    },
    index=[f"joint_{j}" for j in range(7)],
)
err_table.round(4)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 11), height_ratios=[2, 1, 1.2])

ax = axes[0]
for j in range(7):
    ax.plot(t, err[:, j], lw=1.0, label=f"joint_{j}")
for sign in (1, -1):
    ax.axhline(sign * MAX_REL_MOVEMENT, color="red", ls=":", lw=1.2)
ax.text(t[0], -MAX_REL_MOVEMENT, " limiter cap (5°)", color="red", va="bottom", ha="left", fontsize=8)
ax.axhline(0.0, color="grey", ls=":", lw=1.0)
ax.set_ylabel("error [rad]")
ax.set_xlabel("time [s]")
ax.set_title("per-joint tracking error (measured - commanded)")
ax.grid(alpha=0.3)
ax.legend(ncol=7, fontsize=8, loc="upper right")

ax = axes[1]
ax.plot(t, np.linalg.norm(err, axis=1), color="tab:red", lw=1.2)
ax.set_ylabel("||error|| [rad]")
ax.set_xlabel("time [s]")
ax.set_title("error norm over all 7 joints")
ax.grid(alpha=0.3)

ax = axes[2]
x = np.arange(7)
ax.bar(x - 0.2, abs_err.mean(0), width=0.4, color="tab:blue", label="mean |error|")
ax.bar(x + 0.2, abs_err.max(0), width=0.4, color="tab:orange", label="max |error|")
ax.axhline(MAX_REL_MOVEMENT, color="red", ls=":", lw=1.2, label="limiter cap (5°)")
ax.set_xticks(x, [f"joint_{j}" for j in range(7)])
ax.set_ylabel("rad")
ax.set_title("per-joint error summary")
ax.grid(alpha=0.3, axis="y")
ax.legend(fontsize=8)

fig.suptitle(f"Episode {EPISODE_UUID[:8]}: tracking error", y=1.0)
fig.tight_layout()
plt.show()

## Tracking latency

The error above is dominated by lag, not by steady-state offset: the robot reaches the commanded
target a few control steps later. Shifting the measured trace against the command and taking the
mean absolute difference shows how many steps behind the robot runs.

In [ ]:
LAGS = np.arange(0, 13)
lag_err = np.array([np.abs(q_obs[k:] - q_cmd[: len(q_cmd) - k]).mean(0) for k in LAGS])  # (L, 7)

best_overall = int(LAGS[np.argmin(lag_err.mean(1))])
best_per_joint = LAGS[np.argmin(lag_err, axis=0)]
print(f"best global lag: {best_overall} steps ({best_overall / FPS * 1e3:.0f} ms), "
      f"residual mean error {lag_err.mean(1).min():.4f} rad (vs {abs_err.mean():.4f} rad at lag 0)")
print("best lag per joint [steps]:", dict(zip([f"j{j}" for j in range(7)], best_per_joint.tolist())))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for j in range(7):
    ax.plot(LAGS / FPS * 1e3, lag_err[:, j], lw=1.0, marker=".", label=f"joint_{j}")
ax.plot(LAGS / FPS * 1e3, lag_err.mean(1), color="black", lw=2.0, label="mean")
ax.axvline(best_overall / FPS * 1e3, color="grey", ls=":", lw=1.2)
ax.set_xlabel("shift applied to measured trace [ms]")
ax.set_ylabel("mean |error| [rad]")
ax.set_title("error vs. assumed tracking lag")
ax.grid(alpha=0.3)
ax.legend(ncol=2, fontsize=8)

ax = axes[1]
j_worst = int(np.argmax(abs_err.mean(0)))
ax.plot(t, q_cmd[:, j_worst], color="tab:orange", ls="--", lw=1.0, label="commanded")
ax.plot(t, q_obs[:, j_worst], color="tab:green", lw=1.4, label="measured")
ax.plot(t[:len(t) - best_overall], q_obs[best_overall:, j_worst], color="tab:purple", lw=1.2,
        label=f"measured, shifted {best_overall} steps")
ax.set_xlabel("time [s]")
ax.set_ylabel("rad")
ax.set_title(f"joint_{j_worst} (largest mean error) with lag removed")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

## Recorded camera frames

The JPEG frames stored alongside each step, sampled evenly across the episode — the equivalent of
the replay notebook's ZED frame strip, for eyeballing what the robot was doing when the error
spiked.

In [ ]:
cam_names = list(ep["obs"].iloc[0]["frames"].keys())
N_STRIP = 6
strip_idx = np.linspace(0, len(ep) - 1, N_STRIP).astype(int)

fig, axes_cam = plt.subplots(
    len(cam_names), N_STRIP, figsize=(3 * N_STRIP, 2.0 * len(cam_names)), squeeze=False
)
for row, cam in enumerate(cam_names):
    for col, idx in enumerate(strip_idx):
        ax = axes_cam[row, col]
        ax.axis("off")
        img = Image.open(io.BytesIO(ep["obs"].iloc[idx]["frames"][cam]["rgb"]["data"]))
        ax.imshow(np.asarray(img))
        if row == 0:
            ax.set_title(f"t = {t[idx]:.1f} s", fontsize=10)
        if col == 0:
            ax.text(-0.05, 0.5, cam, transform=ax.transAxes, rotation=90,
                    va="center", ha="right", fontsize=11)
fig.suptitle(f"Episode {EPISODE_UUID[:8]}: recorded camera frames", y=1.0)
fig.tight_layout()
plt.show()